<a href="https://colab.research.google.com/github/sourcesync/kagglex_gemma/blob/gw%2Finitial/colab/Rukayat_medical_chatbot2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# This notebook demonstrates:
* A chatbot implementation using Gemma2_instr

# Install required packages

In [1]:
%%time
!pip install -q -U keras-nlp
!pip install -q -U keras>=3
!pip install gradio
!pip install langchain langchain_core
!pip install langchain-google-vertexai
!pip install PyMuPDF sentence-transformers langchain chromadb huggingface-hub
!pip install langchain_community python-docx

CPU times: user 119 ms, sys: 25 ms, total: 144 ms
Wall time: 18.8 s


# Import required packages

In [2]:
import os
import keras
import keras_nlp
from IPython.display import Markdown
import textwrap
import gradio as gr
import langchain_core
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.schema.runnable import Runnable
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
import fitz  # PyMuPDF
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.llms import HuggingFacePipeline
from docx import Document

# Configure this notebook


In [3]:
os.environ["KERAS_BACKEND"] = "jax"  # Or "torch" or "tensorflow".
# Avoid memory fragmentation on JAX backend.
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]="1.00"

# Define some useful functions

In [4]:
def display_chat(prompt, response):
  '''Displays an LLM prompt and response in a pretty way.'''
  prompt = prompt.replace('\n\n','<br><br>')
  prompt = prompt.replace('\n','<br>')
  formatted_prompt = "<font size='+1' color='brown'>🙋‍♂️<blockquote>" + prompt + "</blockquote></font>"
  response = response.replace('•', '  *')
  response = textwrap.indent(response, '', predicate=lambda _: True)
  response = response.replace('\n\n','<br><br>')
  response = response.replace('\n','<br>')
  response = response.replace("```","")
  formatted_text = "<font size='+1' color='teal'>🤖<blockquote>" + response + "</blockquote></font>"
  return Markdown(formatted_prompt+formatted_text)

# A Simple Chatbot Sample

# Load the model

In [5]:
gemma_lm = keras_nlp.models.GemmaCausalLM.from_preset("gemma2_instruct_2b_en")
gemma_lm.summary()

Preprocessor: "gemma_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma_tokenizer (GemmaTokenizer)                              │                      Vocab size: 256,000 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma_backbone                │ (None, None, 2304)        │   2,614,341,888 │ padding_mask[0][0],        │
│ (GemmaBackbone)               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 256000)      │     589,824,000 │ gemma_backbone[0][0]       │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 2,614,341,888 (9.74 GB)

 Trainable params: 2,614,341,888 (9.74 GB)

 Non-trainable params: 0 (0.00 B)

# Define a Chat class which maintains conversation history
* modeled from https://ai.google.dev/gemma/docs/gemma_chat

In [7]:
class ChatState():
  """
  Manages the conversation history for a turn-based chatbot
  Follows the turn-based conversation guidelines for the Gemma family of models
  documented at https://ai.google.dev/gemma/docs/formatting
  """
  __START_TURN_USER__ = "<start_of_turn>user\n" # NOTE: This is only valid for gemma2_instr
  __START_TURN_MODEL__ = "<start_of_turn>model\n" # NOTE: This is only valid for gemma2_instr
  __END_TURN__ = "<end_of_turn>\n" # NOTE: This is only valid for gemma2_instr
  def __init__(self, model, system=""):
    """
    Initializes the chat state.
    Args:
        model: The language model to use for generating responses.
        system: (Optional) System instructions or bot description.
    """
    self.model = model
    self.system = system
    self.history = []
  def add_to_history_as_user(self, message):
    """
    Adds a user message to the history with start/end turn markers.
    """
    self.history.append(self.__START_TURN_USER__ + message + self.__END_TURN__)
  def add_to_history_as_model(self, message):
    """
    Adds a model response to the history with start/end turn markers.
    """
    self.history.append(self.__START_TURN_MODEL__ + message ) #+ self.__END_TURN__)
  def get_history(self):
    """
    Returns the entire chat history as a single string.
    """
    return "".join([*self.history])
  def get_history_blurb(self):
    """
    Returns what to insert into the current prompt
    """
    if len(self.history)==0:
      return ""
    else:
      return \
f"""\n\nUse the following history of your interaction with the user to help answer the question below:\n"""\
f"""{self.get_history()}"""

  def get_full_prompt(self):
    """
    Builds the prompt for the language model, including history and system description.
    """
    prompt = self.get_history() + self.__START_TURN_MODEL__
    if len(self.system)>0:
      prompt = self.system + "\n" + prompt
    return prompt
  def send_message(self, message):
    """
    Handles sending a user message and getting a model response.
    Args:
        message: The user's message.
    Returns:
        The model's response.
    """
    # Step 2: Fake retrieving context from Chroma
    chroma_context = "Some people are allergic to aspirin. "
    # Step 3: Fake retrieving web search context
    web_context = "Many drugs have harmful interactions if taken together. "
    # Step 4: Construct prompt with both Chroma and web search contexts
    prompt = self.get_full_prompt()
    full_prompt = \
f"""You are a highly knowledgeable drug information assistant specializing in drug interactions enquiries. """\
f"""Your goal is to provide an accurate response and relevant information on question below. """\
f"""If you don't know the answer, honestly respond that you don't have the information. """\
f"""Avoid guessing or providing incomplete information.\n\n"""\
f"""Here is an example:\n"""\
f"""Query: Can I take warfarin with ibuprofen?\n"""\
f"""Response: Warfarin and ibuprofen can interact and increase the risk of bleeding. """\
f"""Ibuprofen is a drug with antiplatelet properties and may increase anticoagulation effect of warfarin. """\
f"""It is recommended to avoid using them together or consult your healthcare provider for alternatives.\n\n"""\
f"""Use the following context to answer the question below:\n"""\
f"""{chroma_context}"""\
f"""{web_context}"""\
f"""{self.get_history_blurb()}"""\
f"""\n{self.__START_TURN_USER__ }"""\
f"""{message}"""\
f"""\n{self.__END_TURN__ }"""\
f"""\n{self.__START_TURN_MODEL__ }"""
    # for debugging - print("--->\n" + full_prompt + "<--")
    self.add_to_history_as_user(message)
    # Generate response with full prompt
    response = self.model.generate(full_prompt, max_length=1024)
    # for debugging - print("--->\n" + response + "<--")
    result = response.replace(full_prompt, "")  # Extract only the new response
    # Add the result to chat history
    self.add_to_history_as_model(result)

    return result

# Initialize the Chat object with the model

In [8]:
chat = ChatState(gemma_lm)

# First prompt

In [9]:
message = f"Which drugs would result in severe adverse effect when used with Goserelin?"
display_chat(message, chat.send_message(message))

<font size='+1' color='brown'>🙋‍♂️<blockquote>Which drugs would result in severe adverse effect when used with Goserelin?</blockquote></font><font size='+1' color='teal'>🤖<blockquote>I don't have the information to answer that question. <br><br>It's important to note that drug interactions can be complex and vary depending on individual factors.  **Always consult with a healthcare professional or pharmacist for personalized advice on drug interactions.** <br><end_of_turn></blockquote></font>

# Send a follow-up prompt

In [10]:
message = f"Which drug are we discussing?"
display_chat(message, chat.send_message(message))

<font size='+1' color='brown'>🙋‍♂️<blockquote>Which drug are we discussing?</blockquote></font><font size='+1' color='teal'>🤖<blockquote>The drug we are discussing is **Goserelin**. <br><end_of_turn></blockquote></font>